# DriftSense: Data Generation Pipeline (Phase D1 - D4)

This pipeline synthesizes **DriftBench-v1**:
- **D1**: World demand generation (multi-seasonality, negative-binomial noise, Indian calendar, Poisson deployments) & Infrastructure capacity / autoscaling telemetry across 6 services.
- **D2**: FOCUS pricing model (on-demand, committed, credits/refunds, contracted costs, partitioned parquet).
- **D3**: Fault injection at root layer (6 fault types, decoys, overlapping, confounded scenarios) & `ground_truth` generation.
- **D4**: Realistic data dirt injection (missing hours, duplicates, late restatements, timezone shifts, unit mix) & `dirt_log`.

### 1. Setup & Environment

In [ ]:
# Mount Drive for Colab persistent artifact storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import sys
from pathlib import Path

current_dir = Path.cwd()
if (current_dir / 'src').exists():
    sys.path.insert(0, str(current_dir / 'src'))
elif (current_dir.parent / 'src').exists():
    sys.path.insert(0, str(current_dir.parent / 'src'))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import driftsense
from driftsense.sim.world import generate_world_timeline
from driftsense.sim.infra import generate_infra_telemetry
from driftsense.sim.pricing import apply_pricing_model, save_partitioned_billed_parquet, FOCUS_COLUMNS
from driftsense.utils.seed import set_seed
from driftsense.utils.io import load_config
from driftsense.utils.summary import print_result_summary

print(f"DriftSense Version: {driftsense.__version__}")

### 2. Configuration & Scale Selection

In [ ]:
import time
start_time = time.time()

base_cfg = load_config("base.yaml")
scale_cfg = load_config("scale.yaml")

# Select scale: 'tiny' for smoke/tests, 'small' for quick Colab verification, 'full' for 365-day benchmark
SCALE = "tiny"  # Switch to 'full' for production run in Colab
active_params = scale_cfg[SCALE]

SEED = base_cfg.get("seed", 42)
set_seed(SEED)

DAYS = active_params["days"]
PROVIDERS = base_cfg.get("providers", ["aws_like", "azure_like"])
RES_PER_PROVIDER = active_params["resources_per_provider"]

print(f"Active Scale: {SCALE} | Days: {DAYS} | Providers: {PROVIDERS} | Resources/Provider: {RES_PER_PROVIDER}")

### 3. Run: World, Infrastructure & Pricing Simulation (D1 & D2)

In [ ]:
print("Generating world timeline (D1)...")
world_df, events_df = generate_world_timeline(
    days=DAYS,
    start_date_str=base_cfg["date_range"]["start"],
    annual_growth_rate=0.35,
    seed=SEED,
)

print("Generating infrastructure telemetry (D1)...")
infra_df = generate_infra_telemetry(
    world_df=world_df,
    providers=PROVIDERS,
    resources_per_provider=RES_PER_PROVIDER,
    seed=SEED,
)

print("Applying FOCUS pricing and credits model (D2)...")
billed_df, updated_events_df = apply_pricing_model(
    infra_df=infra_df,
    events_df=events_df,
    credit_rate=0.005,
    seed=SEED,
)

print(f"World timeline shape: {world_df.shape}")
print(f"Infra telemetry shape: {infra_df.shape}")
print(f"Billed records shape: {billed_df.shape}")
print(f"Events logged: {len(updated_events_df)}")

### 4. Inspect: Telemetry Visualizations & Cost Distributions

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 13))

# 1. Web Traffic (Hourly Requests)
ts_world = world_df["timestamp"].to_numpy()
req_world = world_df["traffic_requests"].to_numpy()
limit_idx = min(len(ts_world), 24 * 14)
axes[0].plot(ts_world[:limit_idx], req_world[:limit_idx], color="royalblue", lw=1.2)
axes[0].set_title("World Web Traffic (Hourly Requests - Diurnal & Weekly Seasonality)", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Requests / hr")
axes[0].grid(True, alpha=0.3)

# 2. Hourly Billed Cost by Service
service_cost_ts = billed_df.group_by(["ChargePeriodStart", "ServiceName"]).agg(pl.col("BilledCost").sum()).sort("ChargePeriodStart")
for s in service_cost_ts["ServiceName"].unique():
    sub_s = service_cost_ts.filter(pl.col("ServiceName") == s).head(24 * 14)
    axes[1].plot(sub_s["ChargePeriodStart"].to_numpy(), sub_s["BilledCost"].to_numpy(), label=s, lw=1.2)
axes[1].set_title("Hourly Billed Cost by Service (INR)", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Cost (INR)")
axes[1].legend(loc="upper right", ncol=3)
axes[1].grid(True, alpha=0.3)

# 3. Dynamic Instance Counts (Compute & Kubernetes Autoscalers)
compute_sample = infra_df.filter(pl.col("ServiceName") == "compute").head(24 * 14)
axes[2].step(compute_sample["ChargePeriodStart"].to_numpy(), compute_sample["x_instance_count"].to_numpy(), color="darkorange", where="post", lw=1.5)
axes[2].set_title("Dynamic Instance Count (Autoscaler with Cooldown)", fontsize=11, fontweight="bold")
axes[2].set_ylabel("Instance Count")
axes[2].grid(True, alpha=0.3)

# 4. Cost Invariant Check: List vs Contracted vs Effective
sample_res = billed_df.filter((pl.col("ServiceName") == "compute") & (pl.col("ChargeCategory") == "Usage")).head(24 * 14)
axes[3].plot(sample_res["ChargePeriodStart"].to_numpy(), sample_res["ListCost"].to_numpy(), label="ListCost", color="gray", linestyle=":", lw=1.5)
axes[3].plot(sample_res["ChargePeriodStart"].to_numpy(), sample_res["ContractedCost"].to_numpy(), label="ContractedCost (EDP)", color="blue", lw=1.2)
axes[3].plot(sample_res["ChargePeriodStart"].to_numpy(), sample_res["EffectiveCost"].to_numpy(), label="EffectiveCost (Amortized)", color="green", lw=1.2)
axes[3].set_title("FOCUS Pricing Tier Verification (ListCost >= ContractedCost >= EffectiveCost)", fontsize=11, fontweight="bold")
axes[3].set_ylabel("Hourly Rate (INR)")
axes[3].legend(loc="upper right")
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5. Checks: Acceptance Criteria Verification
- Cost identity: ListCost >= ContractedCost >= EffectiveCost
- Credit & refund line items flagged with negative amounts
- Column set matches FOCUS specification
- Storage monotone & Zero NaNs

In [ ]:
checks_status = {}

# 1. FOCUS columns match
non_part_cols = [c for c in billed_df.columns if c != "billing_month"]
assert set(non_part_cols) == set(FOCUS_COLUMNS), f"Column mismatch: {set(non_part_cols) ^ set(FOCUS_COLUMNS)}"
checks_status["focus_columns_exact"] = "PASS"

# 2. Cost identity: List >= Contracted >= Effective
usage_rows = billed_df.filter(pl.col("ChargeCategory") == "Usage")
list_arr = usage_rows["ListCost"].to_numpy()
contracted_arr = usage_rows["ContractedCost"].to_numpy()
effective_arr = usage_rows["EffectiveCost"].to_numpy()
tol = 1e-2
assert (list_arr + tol >= contracted_arr).all(), "ListCost < ContractedCost violation"
assert (contracted_arr + tol >= effective_arr).all(), "ContractedCost < EffectiveCost violation"
checks_status["cost_identity_valid"] = "PASS"

# 3. Credits and refunds present and negative
credit_rows = billed_df.filter(pl.col("ChargeCategory").is_in(["Credit", "Refund"]))
assert len(credit_rows) > 0, "No credits generated"
assert (credit_rows["BilledCost"] < 0).all()
assert (credit_rows["ChargeClass"] == "Correction").all()
checks_status["credits_refunds_flagged"] = "PASS"

# 4. Storage monotonicity
storage_res = infra_df.filter(pl.col("ServiceName") == "object_storage")["ResourceId"].unique().to_list()[0]
storage_arr = infra_df.filter(pl.col("ResourceId") == storage_res)["x_storage_gb"].to_numpy()
assert (np.diff(storage_arr) >= -1e-6).all()
checks_status["storage_monotone"] = "PASS"

# 5. Zero NaNs
assert all(billed_df[col].null_count() == 0 for col in billed_df.columns)
checks_status["zero_nans"] = "PASS"

print("Acceptance Checks Status:", checks_status)

### 6. Save Partitioned Parquet & Artifacts

In [ ]:
out_dir = Path("artifacts/d2")
out_dir.mkdir(parents=True, exist_ok=True)

raw_data_dir = "data/raw"
saved_partitions = save_partitioned_billed_parquet(billed_df, output_dir=raw_data_dir)

events_path = out_dir / "events.parquet"
updated_events_df.write_parquet(events_path)

artifacts_saved = saved_partitions + [str(events_path)]
print(f"Successfully saved {len(saved_partitions)} monthly partitions to {raw_data_dir}.")
print(f"Events saved to {events_path}.")

### 7. RESULT SUMMARY

In [ ]:
runtime = time.time() - start_time

summary = {
    "phase": "D2",
    "scale": SCALE,
    "seed": SEED,
    "runtime_sec": runtime,
    "shapes": {
        "world_df": f"{world_df.height} x {world_df.width}",
        "infra_df": f"{infra_df.height} x {infra_df.width}",
        "billed_df": f"{billed_df.height} x {billed_df.width}",
        "events_df": f"{updated_events_df.height} x {updated_events_df.width}",
    },
    "key_metrics": {
        "total_billed_cost_inr": float(round(billed_df["BilledCost"].sum(), 2)),
        "total_credits_inr": float(round(credit_rows["BilledCost"].sum(), 2)),
        "num_credit_records": len(credit_rows),
        "num_partitions": len(saved_partitions),
    },
    "checks": checks_status,
    "warnings": [],
    "artifacts": artifacts_saved[:5] + ["..."]
}

print_result_summary(summary)